In [7]:
from pathlib import Path
import pandas as pd

# ---------------------------------------------------------
# STEP 1: Enter the exact location of your 01_raw_data folder
# ---------------------------------------------------------

RAW_DATA_DIR = Path(
    r"D:\NHS Elective Care Performance & Backlog Recovery Analytics\01_raw_data"
)

# ---------------------------------------------------------
# STEP 2: Check whether the folder exists
# ---------------------------------------------------------

print("Raw data folder:", RAW_DATA_DIR)
print("Folder exists:", RAW_DATA_DIR.exists())
print("-" * 80)

# Stop the code if the folder path is incorrect
if not RAW_DATA_DIR.exists():
    raise FileNotFoundError(
        "The 01_raw_data folder was not found. "
        "Check the folder path entered in RAW_DATA_DIR."
    )

# ---------------------------------------------------------
# STEP 3: Find all Excel files in the folder
# ---------------------------------------------------------

excel_files = sorted(RAW_DATA_DIR.glob("*.xlsx"))

print(f"Number of Excel files found: {len(excel_files)}")
print("-" * 80)

# ---------------------------------------------------------
# STEP 4: Prepare variables for validation
# ---------------------------------------------------------

reference_columns = None
reference_file = None
problems = []
successful_files = 0

# ---------------------------------------------------------
# STEP 5: Check each monthly Excel file
# ---------------------------------------------------------

for file_path in excel_files:

    try:
        # Read the workbook structure and sheet names
        workbook = pd.ExcelFile(file_path)

        # Find the worksheet named Provider
        provider_sheet = next(
            (
                sheet
                for sheet in workbook.sheet_names
                if sheet.strip().lower() == "provider"
            ),
            None
        )

        # Record a problem if the Provider sheet is missing
        if provider_sheet is None:
            problems.append(
                f"{file_path.name}: Provider worksheet not found"
            )
            print(f"FAILED: {file_path.name} — Provider sheet not found")
            continue

        # Read only the column headings
        # header=13 means Excel row 14 is treated as the header
        header_data = pd.read_excel(
            file_path,
            sheet_name=provider_sheet,
            header=13,
            nrows=0
        )

        # Remove unnecessary spaces from column names
        columns = [
            str(column).strip()
            for column in header_data.columns
        ]

        # The first workbook becomes the reference structure
        if reference_columns is None:
            reference_columns = columns
            reference_file = file_path.name
            successful_files += 1

            print(
                f"REFERENCE: {file_path.name} — "
                f"{len(columns)} columns"
            )

        # Compare all later files with the reference workbook
        elif columns != reference_columns:
            problems.append(
                f"{file_path.name}: Column structure differs "
                f"from {reference_file}"
            )

            print(
                f"DIFFERENT: {file_path.name} — "
                f"{len(columns)} columns"
            )

        else:
            successful_files += 1

            print(
                f"OK: {file_path.name} — "
                f"{len(columns)} columns"
            )

    except Exception as error:
        problems.append(
            f"{file_path.name}: {error}"
        )

        print(
            f"ERROR: {file_path.name} — {error}"
        )

# ---------------------------------------------------------
# STEP 6: Display the final validation result
# ---------------------------------------------------------

print("-" * 80)
print(f"Files found: {len(excel_files)}")
print(f"Files that passed: {successful_files}")
print(f"Problems found: {len(problems)}")

if len(excel_files) != 25:
    print(
        f"\nWARNING: Expected 25 monthly files, "
        f"but found {len(excel_files)}."
    )

elif problems:
    print("\nThe following problems need to be checked:")

    for problem in problems:
        print(f"- {problem}")

else:
    print(
        "\nAll 25 monthly files passed "
        "the initial structure check."
    )

Raw data folder: D:\NHS Elective Care Performance & Backlog Recovery Analytics\01_raw_data
Folder exists: True
--------------------------------------------------------------------------------
Number of Excel files found: 25
--------------------------------------------------------------------------------
REFERENCE: Incomplete-Provider-Apr24-XLSX-8M-revised.xlsx — 119 columns
DIFFERENT: Incomplete-Provider-Apr25-XLSX-9M-revised.xlsx — 120 columns
DIFFERENT: Incomplete-Provider-Apr26-XLSX-9M-X7gGnn.xlsx — 120 columns
OK: Incomplete-Provider-Aug24-XLSX-9M-revised.xlsx — 119 columns
DIFFERENT: Incomplete-Provider-Aug25-XLSX-9M-revised-2.xlsx — 120 columns
OK: Incomplete-Provider-Dec24-XLSX-9M-revised.xlsx — 119 columns
DIFFERENT: Incomplete-Provider-Dec25-XLSX-9M-6jPlxd.xlsx — 120 columns
OK: Incomplete-Provider-Feb25-XLSX-9M-revised.xlsx — 119 columns
DIFFERENT: Incomplete-Provider-Feb26-XLSX-9M-9j03fJT.xlsx — 120 columns
OK: Incomplete-Provider-Jan25-XLSX-9M-revised.xlsx — 119 columns
DIF

In [9]:
# Compare the final 119-column file with the first 120-column file

feb25_file = next(
    file for file in excel_files
    if "feb25" in file.name.lower()
)

mar25_file = next(
    file for file in excel_files
    if "mar25" in file.name.lower()
)


def get_provider_columns(file_path):
    workbook = pd.ExcelFile(file_path)

    provider_sheet = next(
        sheet
        for sheet in workbook.sheet_names
        if sheet.strip().lower() == "provider"
    )

    header_data = pd.read_excel(
        file_path,
        sheet_name=provider_sheet,
        header=13,
        nrows=0
    )

    return [
        str(column).strip()
        for column in header_data.columns
    ]


feb25_columns = get_provider_columns(feb25_file)
mar25_columns = get_provider_columns(mar25_file)

added_columns = [
    column
    for column in mar25_columns
    if column not in feb25_columns
]

removed_columns = [
    column
    for column in feb25_columns
    if column not in mar25_columns
]

print("February 2025 file:", feb25_file.name)
print("February 2025 columns:", len(feb25_columns))

print("\nMarch 2025 file:", mar25_file.name)
print("March 2025 columns:", len(mar25_columns))

print("\nColumns added in March 2025:")
for column in added_columns:
    position = mar25_columns.index(column) + 1
    print(f"- Column {position}: {column}")

print("\nColumns removed after February 2025:")
if removed_columns:
    for column in removed_columns:
        print(f"- {column}")
else:
    print("- None")

common_order_before = [
    column for column in feb25_columns
    if column in mar25_columns
]

common_order_after = [
    column for column in mar25_columns
    if column in feb25_columns
]

print(
    "\nExisting columns remain in the same order:",
    common_order_before == common_order_after
)

February 2025 file: Incomplete-Provider-Feb25-XLSX-9M-revised.xlsx
February 2025 columns: 119

March 2025 file: Incomplete-Provider-Mar25-XLSX-9M-revised.xlsx
March 2025 columns: 120

Columns added in March 2025:
- Column 120: % 52 plus weeks

Columns removed after February 2025:
- None

Existing columns remain in the same order: True


In [12]:
monthly_data = []

for file_path in excel_files:
    data = pd.read_excel(
        file_path,
        sheet_name="Provider",
        header=13
    )

    monthly_data.append(data)

print("Files loaded:", len(monthly_data))
print("Rows in first file:", len(monthly_data[0]))
print("Columns in first file:", len(monthly_data[0].columns))

Files loaded: 25
Rows in first file: 3768
Columns in first file: 119


In [14]:
import re

reporting_months = []

for file_path in excel_files:
    match = re.search(
        r"-(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)(\d{2})-",
        file_path.name,
        re.IGNORECASE
    )

    if match is None:
        raise ValueError(
            f"Reporting month could not be found in: {file_path.name}"
        )

    month_text = match.group(1) + match.group(2)

    reporting_month = pd.to_datetime(
        month_text,
        format="%b%y"
    )

    reporting_months.append(reporting_month)

print("Reporting months found:", len(reporting_months))
print("Earliest month:", min(reporting_months))
print("Latest month:", max(reporting_months))
print("Unique months:", len(set(reporting_months)))

Reporting months found: 25
Earliest month: 2024-04-01 00:00:00
Latest month: 2026-04-01 00:00:00
Unique months: 25


In [16]:
for data, reporting_month in zip(monthly_data, reporting_months):
    data["Reporting_Month"] = reporting_month

print("Reporting_Month added to:", len(monthly_data), "datasets")

for i in range(3):
    print(
        excel_files[i].name,
        "→",
        monthly_data[i]["Reporting_Month"].iloc[0].date()
    )

Reporting_Month added to: 25 datasets
Incomplete-Provider-Apr24-XLSX-8M-revised.xlsx → 2024-04-01
Incomplete-Provider-Apr25-XLSX-9M-revised.xlsx → 2025-04-01
Incomplete-Provider-Apr26-XLSX-9M-X7gGnn.xlsx → 2026-04-01


In [18]:
april24_index = reporting_months.index(
    pd.Timestamp("2024-04-01")
)

april24_data = monthly_data[april24_index]

columns_to_check = [
    "Total number of incomplete pathways",
    "Total 52 plus weeks"
]

print(april24_data[columns_to_check].head())

print("\nData types:")
print(april24_data[columns_to_check].dtypes)

print(
    "\nPercentage column present:",
    "% 52 plus weeks" in april24_data.columns
)

   Total number of incomplete pathways  Total 52 plus weeks
0                                 3035                  244
1                                 6179                  474
2                                 7707                  439
3                                 6990                  882
4                                 4968                   37

Data types:
Total number of incomplete pathways    int64
Total 52 plus weeks                    int64
dtype: object

Percentage column present: False


In [20]:
updated_datasets = 0

for data in monthly_data:
    if "% 52 plus weeks" not in data.columns:
        total_pathways = data["Total number of incomplete pathways"]

        data["% 52 plus weeks"] = (
            data["Total 52 plus weeks"] / total_pathways
        ).where(total_pathways != 0)

        updated_datasets += 1

print("Datasets updated:", updated_datasets)
print(
    "Percentage column now present in all datasets:",
    all("% 52 plus weeks" in data.columns for data in monthly_data)
)

Datasets updated: 11
Percentage column now present in all datasets: True


In [22]:
combined_data = pd.concat(
    monthly_data,
    ignore_index=True
)

print("Total rows:", len(combined_data))
print("Total columns:", len(combined_data.columns))
print(
    "Reporting months:",
    combined_data["Reporting_Month"].nunique()
)

Total rows: 91512
Total columns: 121
Reporting months: 25


In [24]:
key_columns = [
    "Provider Code",
    "Provider Name",
    "Treatment Function Code",
    "Treatment Function",
    "Reporting_Month"
]

print(combined_data[key_columns].isna().sum())

Provider Code              0
Provider Name              0
Treatment Function Code    0
Treatment Function         0
Reporting_Month            0
dtype: int64


In [26]:
kpi_columns = [
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Total 52 plus weeks",
    "% 52 plus weeks"
]

print(combined_data[kpi_columns].isna().sum())

Total number of incomplete pathways        0
Total within 18 weeks                      0
% within 18 weeks                          0
Total 52 plus weeks                        0
% 52 plus weeks                        12276
dtype: int64


In [28]:
missing_52_pct = combined_data[
    combined_data["% 52 plus weeks"].isna()
]

print("Missing percentage rows:", len(missing_52_pct))

print(
    "Rows with zero total pathways:",
    (missing_52_pct["Total number of incomplete pathways"] == 0).sum()
)

print(
    "Rows with pathways but zero 52-week waits:",
    (
        (missing_52_pct["Total number of incomplete pathways"] > 0)
        & (missing_52_pct["Total 52 plus weeks"] == 0)
    ).sum()
)

print(
    "Rows requiring investigation:",
    (
        (missing_52_pct["Total number of incomplete pathways"] > 0)
        & (missing_52_pct["Total 52 plus weeks"] > 0)
    ).sum()
)

Missing percentage rows: 12276
Rows with zero total pathways: 12276
Rows with pathways but zero 52-week waits: 0
Rows requiring investigation: 0


In [30]:
record_key = [
    "Reporting_Month",
    "Provider Code",
    "Treatment Function Code"
]

duplicate_records = combined_data.duplicated(
    subset=record_key,
    keep=False
)

print("Duplicate key rows:", duplicate_records.sum())

Duplicate key rows: 0


In [32]:
for number, column in enumerate(combined_data.columns, start=1):
    print(number, column)

1 Unnamed: 0
2 Region Code
3 Provider Code
4 Provider Name
5 Treatment Function Code
6 Treatment Function
7 >0-1
8 >1-2
9 >2-3
10 >3-4
11 >4-5
12 >5-6
13 >6-7
14 >7-8
15 >8-9
16 >9-10
17 >10-11
18 >11-12
19 >12-13
20 >13-14
21 >14-15
22 >15-16
23 >16-17
24 >17-18
25 >18-19
26 >19-20
27 >20-21
28 >21-22
29 >22-23
30 >23-24
31 >24-25
32 >25-26
33 >26-27
34 >27-28
35 >28-29
36 >29-30
37 >30-31
38 >31-32
39 >32-33
40 >33-34
41 >34-35
42 >35-36
43 >36-37
44 >37-38
45 >38-39
46 >39-40
47 >40-41
48 >41-42
49 >42-43
50 >43-44
51 >44-45
52 >45-46
53 >46-47
54 >47-48
55 >48-49
56 >49-50
57 >50-51
58 >51-52
59 >52-53
60 >53-54
61 >54-55
62 >55-56
63 >56-57
64 >57-58
65 >58-59
66 >59-60
67 >60-61
68 >61-62
69 >62-63
70 >63-64
71 >64-65
72 >65-66
73 >66-67
74 >67-68
75 >68-69
76 >69-70
77 >70-71
78 >71-72
79 >72-73
80 >73-74
81 >74-75
82 >75-76
83 >76-77
84 >77-78
85 >78-79
86 >79-80
87 >80-81
88 >81-82
89 >82-83
90 >83-84
91 >84-85
92 >85-86
93 >86-87
94 >87-88
95 >88-89
96 >89-90
97 >90-91
98 >91

In [34]:
combined_data.drop(
    columns=["Unnamed: 0"],
    inplace=True
)

print("Total columns after removal:", len(combined_data.columns))
print("Unnamed: 0 still present:", "Unnamed: 0" in combined_data.columns)

Total columns after removal: 120
Unnamed: 0 still present: False


In [36]:
for number, column in enumerate(combined_data.columns[-20:], start=1):
    print(number, column)

1 >95-96
2 >96-97
3 >97-98
4 >98-99
5 >99-100
6 >100-101
7 >101-102
8 >102-103
9 >103-104
10 104 plus
11 Total number of incomplete pathways
12 Total within 18 weeks
13 % within 18 weeks
14 Average (median) waiting time (in weeks)
15 92nd percentile waiting time (in weeks)
16 Total 52 plus weeks
17 Total 78 plus weeks
18 Total 65 plus weeks
19 Reporting_Month
20 % 52 plus weeks


In [38]:
analysis_columns = [
    "Region Code",
    "Provider Code",
    "Provider Name",
    "Treatment Function Code",
    "Treatment Function",
    "Total number of incomplete pathways",
    "Total within 18 weeks",
    "% within 18 weeks",
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)",
    "Total 52 plus weeks",
    "Total 65 plus weeks",
    "Total 78 plus weeks",
    "% 52 plus weeks",
    "Reporting_Month"
]

rtt_summary_data = combined_data[analysis_columns].copy()

print("Rows:", len(rtt_summary_data))
print("Columns:", len(rtt_summary_data.columns))

Rows: 91512
Columns: 15


In [40]:
print(rtt_summary_data.dtypes)

Region Code                                         object
Provider Code                                       object
Provider Name                                       object
Treatment Function Code                             object
Treatment Function                                  object
Total number of incomplete pathways                  int64
Total within 18 weeks                                int64
% within 18 weeks                                  float64
Average (median) waiting time (in weeks)            object
92nd percentile waiting time (in weeks)             object
Total 52 plus weeks                                  int64
Total 65 plus weeks                                  int64
Total 78 plus weeks                                  int64
% 52 plus weeks                                    float64
Reporting_Month                             datetime64[ns]
dtype: object


In [42]:
waiting_time_columns = [
    "Average (median) waiting time (in weeks)",
    "92nd percentile waiting time (in weeks)"
]

for column in waiting_time_columns:
    non_numeric_values = pd.to_numeric(
        rtt_summary_data[column],
        errors="coerce"
    ).isna() & rtt_summary_data[column].notna()

    print(column)
    print(rtt_summary_data.loc[non_numeric_values, column].value_counts())
    print()

Average (median) waiting time (in weeks)
Average (median) waiting time (in weeks)
-    29487
Name: count, dtype: int64

92nd percentile waiting time (in weeks)
92nd percentile waiting time (in weeks)
-    29487
Name: count, dtype: int64



In [44]:
dash_rows = rtt_summary_data[
    rtt_summary_data[
        "Average (median) waiting time (in weeks)"
    ] == "-"
]

print("Rows containing dash:", len(dash_rows))

print(
    "Dash rows with zero pathways:",
    (
        dash_rows["Total number of incomplete pathways"] == 0
    ).sum()
)

print(
    "Dash rows with positive pathways:",
    (
        dash_rows["Total number of incomplete pathways"] > 0
    ).sum()
)

Rows containing dash: 29487
Dash rows with zero pathways: 26913
Dash rows with positive pathways: 2574


In [46]:
positive_dash_rows = dash_rows[
    dash_rows["Total number of incomplete pathways"] > 0
]

print(
    "Minimum pathways:",
    positive_dash_rows["Total number of incomplete pathways"].min()
)

print(
    "Maximum pathways:",
    positive_dash_rows["Total number of incomplete pathways"].max()
)

print(
    "Rows with 20 or more pathways:",
    (
        positive_dash_rows["Total number of incomplete pathways"] >= 20
    ).sum()
)

Minimum pathways: 1
Maximum pathways: 19
Rows with 20 or more pathways: 0


In [48]:
for column in waiting_time_columns:
    rtt_summary_data[column] = pd.to_numeric(
        rtt_summary_data[column],
        errors="coerce"
    )

print(rtt_summary_data[waiting_time_columns].dtypes)

Average (median) waiting time (in weeks)    float64
92nd percentile waiting time (in weeks)     float64
dtype: object


In [50]:
CLEAN_DATA_DIR = RAW_DATA_DIR.parent / "04_clean_data"
CLEAN_DATA_DIR.mkdir(exist_ok=True)

output_file = CLEAN_DATA_DIR / "clean_rtt_summary_data.csv"

rtt_summary_data.to_csv(
    output_file,
    index=False
)

print("File saved:", output_file)
print("Rows exported:", len(rtt_summary_data))
print("Columns exported:", len(rtt_summary_data.columns))

File saved: D:\NHS Elective Care Performance & Backlog Recovery Analytics\04_clean_data\clean_rtt_summary_data.csv
Rows exported: 91512
Columns exported: 15


In [52]:
export_check = pd.read_csv(
    output_file,
    parse_dates=["Reporting_Month"]
)

print("Rows:", len(export_check))
print("Columns:", len(export_check.columns))
print("Reporting months:", export_check["Reporting_Month"].nunique())
print("Earliest month:", export_check["Reporting_Month"].min())
print("Latest month:", export_check["Reporting_Month"].max())

Rows: 91512
Columns: 15
Reporting months: 25
Earliest month: 2024-04-01 00:00:00
Latest month: 2026-04-01 00:00:00
